In [ ]:
using DifferentialEquations
using Plots
using LinearAlgebra

In [ ]:
function advection_mf!(du, u, p, t)
    c = p[1];
    dx = p[2];
    x = p[3];
    g = p[4];
    f = p[5];
    
    nx = length(u);

    # populate the entry at x1
    du[1] = -c/dx * u[1]  + f(x[1], t) + c/dx * g(t);
    # populate the rest of the entries
    for i = 2:nx
        du[i] = -c/dx * (u[i]-u[i-1]) + f(x[i],t);
    end
    
    du
end

In [ ]:
x = LinRange(-5.0, 5.0, 51)[2:end];

u0 = exp.(-x.^2) # initial condition
tspan = (0.0, 5.0) # (t0, tmax)

c = 1;  # speed
dx = x[2] - x[1];
g = t-> cos(pi*t); # boundary condition
f = (x,t) -> 0.1 * sin(pi * x); # source/sink


In [ ]:
# store parameters
p = (c, dx, x, g, f);
# define and solve problem
prob_mf = ODEProblem(advection_mf!, u0, tspan, p)
sol_mf = solve(prob_mf);    


In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 51);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol_mf(t_plot[i]), label="t=$(t_plot[i])")
    ylims!(-1, 1)
end

In [ ]:
gif(anim, "advection.gif", fps=15)

In [ ]:
function backward_difference_matrix(nx, dx)
    D = zeros(nx, nx)

    for i in 1:nx
        D[i, i] = 1/dx
        if i > 1
            D[i, i-1] = -1/dx
        end
    end

    return D
end

In [ ]:
function advection_dense!(du, u, p, t)
    c = p[1];
    dx = p[2];
    x = p[3];
    g = p[4];
    f = p[5];
    Dx = p[6];

    # du .= -c * Dx * u + f.(x,t);
    # du .= -c * Dx * u;
    mul!(du, Dx, u);
    @. du *= -c;
    @. du += f(x,t);
    # du .+= f.(x,t);
    # edit to capture the boundary condition
    du[1] += c/dx * g(t);
        
    du
end


In [ ]:
x = LinRange(-5.0, 5.0, 51)[2:end];

u0 = exp.(-x.^2) # initial condition
tspan = (0.0, 5.0) # (t0, tmax)

c = 1;  # speed
dx = x[2] - x[1];
g = t-> cos(pi*t); # boundary condition
f = (x,t) -> 0.1 * sin(pi * x); # source/sink
nx = length(x);
Dx = backward_difference_matrix(nx, dx)

# store parameters as a tuple
p = (c, dx, x, g, f, Dx);
# define and solve problem
prob_dense = ODEProblem(advection_dense!, u0, tspan, p)
sol_dense = solve(prob_dense);    

In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 51);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol_dense(t_plot[i]), label="t=$(t_plot[i])")
    ylims!(-1, 1)
end

gif(anim, "advection.gif", fps=15)

In [ ]:
norm(sol_mf.u[end] - sol_dense.u[end])

In [ ]:
using BenchmarkTools

In [ ]:
x = LinRange(-5.0, 5.0, 400+1)[2:end];

u0 = exp.(-x.^2) # initial condition
c = 1;  # speed
dx = x[2] - x[1];
g = t-> cos(pi*t); # boundary condition
f = (x,t) -> 0.1 * sin(pi * x); # source/sink
nx = length(x);
Dx = backward_difference_matrix(nx, dx)
du = zeros(nx);

p_mf = (c, dx, x, g, f);
p_dense = (c, dx, x, g, f, Dx);

t0 = 0.0;

@show nx;
@btime advection_mf!($(du), $(u0), $(p_mf), $(t0));
@btime advection_dense!($(du), $(u0), $(p_dense), $(t0));

In [ ]:
@btime advection_mf!(du, u0, p, tspan[1]);

In [ ]:
using SparseArrays

In [ ]:
function sparse_backward_difference_matrix(nx, dx)
    main = fill(1.0 / dx, nx)
    sub  = fill(-1.0 / dx, nx-1)
    return spdiagm(0 => main, -1 => sub)
end

In [ ]:
sparse_backward_difference_matrix(50, 0.1)

In [ ]:
function advection_sparse!(du, u, p, t)
    c = p[1];
    dx = p[2];
    x = p[3];
    g = p[4];
    f = p[5];
    Dx = p[6];

    # du .= -c * Dx * u + f.(x,t);
    # du .= -c * Dx * u;
    mul!(du, Dx, u);
    @. du *= -c;
    @. du += f(x,t);
    # du .+= f.(x,t);
    # edit to capture the boundary condition
    du[1] += c/dx * g(t);
        
    du
end


In [ ]:
advection_sparse!(du, u0, p_sparse, t0);

In [ ]:
x = LinRange(-5.0, 5.0, 200+1)[2:end];

u0 = exp.(-x.^2) # initial condition
c = 1;  # speed
dx = x[2] - x[1];
g = t-> cos(pi*t); # boundary condition
f = (x,t) -> 0.1 * sin(pi * x); # source/sink
nx = length(x);
#Dx = backward_difference_matrix(nx, dx);
Dx = sparse_backward_difference_matrix(nx, dx);
du = zeros(nx);

p_mf = (c, dx, x, g, f);
p_sparse = (c, dx, x, g, f, Dx);

t0 = 0.0;

@show nx;
@btime advection_mf!($(du), $(u0), $(p_mf), $(t0));
@btime advection_sparse!($(du), $(u0), $(p_sparse), $(t0));

In [ ]:
Matrix(Dx)

In [ ]:
A = spdiagm(0 => -2*ones(4), 1=>ones(3), -1=>ones(3))

In [ ]:
for j in 1:A.n
           for k in A.colptr[j]:A.colptr[j+1]-1
               println("A[$(A.rowval[k]),$(j)] = $( A.nzval[k])")
           end
       end

In [ ]:
function sparse_backward_difference_matrix_pbc(nx, dx)
    main = fill(1.0 / dx, nx)
    sub  = fill(-1.0 / dx, nx-1)
    Dx = spdiagm(0 => main, -1 => sub);
    Dx[1, end] = -1.0 / dx
    return Dx
end

In [ ]:
sparse_backward_difference_matrix_pbc(50, 0.1)

In [ ]:
function advection_pbc!(du, u, p, t)
    c = p[1];
    x = p[2];
    f = p[3];
    Dx = p[4];

    du .= -c * Dx * u + f.(x,t);        
    du
end

In [ ]:
x = LinRange(-5.0, 5.0, 101)[1:end-1];

u0 = exp.(-x.^2) # initial condition
tspan = (0.0, 10.0) # (t0, tmax)

c = 1;  # speed
dx = x[2] - x[1];
f = (x,t) -> 0.0; # source/sink
nx = length(x);
Dx = sparse_backward_difference_matrix_pbc(nx, dx)

# store parameters as a tuple
p = (c, x, f, Dx);
# define and solve problem
prob_pbc = ODEProblem(advection_pbc!, u0, tspan, p)
sol_pbc = solve(prob_pbc);    

In [ ]:
sol_pbc.alg

In [ ]:
plot(x, sol_pbc(tspan[1]), label="t=$(tspan[1])")
plot!(x, sol_pbc(tspan[2]), label="t=$(tspan[2])")

In [ ]:
x = LinRange(-5.0, 5.0, 800 + 1)[1:end-1];

u0 = exp.(-x.^2) # initial condition
tspan = (0.0, 10.0) # (t0, tmax)

c = 1;  # speed
dx = x[2] - x[1];
f = (x,t) -> 0.0; # source/sink
nx = length(x);
Dx = sparse_backward_difference_matrix_pbc(nx, dx)

dt = 0.01;

# store parameters as a tuple
p = (c, x, f, Dx);
# define and solve problem
prob_pbc = ODEProblem(advection_pbc!, u0, tspan, p)
sol_pbc = solve(prob_pbc,Euler(); dt=dt, adaptive=false);

plot(x, sol_pbc(tspan[1]), label="t=$(tspan[1])")
plot!(x, sol_pbc(tspan[2]), label="t=$(tspan[2])")

In [ ]:
t_plot = LinRange(tspan[1], tspan[2], 51);

anim = @animate for i in 1:length(t_plot)
    plot(x, sol_pbc(t_plot[i]), label="t=$(t_plot[i])")
    ylims!(-1, 1)
end

gif(anim, "advection.gif", fps=15)